# Replication Specification

## 1. Objective

Estimate weekly Managed Money positioning for six energy futures markets from price-derived trend signals, following Insight 177. The first model layer is common across markets; the second layer allocates the extracted signals separately by market and includes a regularized time-varying bias.

## 2. Data contract

Required columns:

- Prices: `market`, `date`, `nearby`, `PX_SETTLE`.
- Metadata: `market`, `maturity_year`, `maturity_month`, `LAST_TRADEABLE_DT`.
- Managed Money: `market`, `basis`, `report_week_tuesday`, `mm_net`.

Default Managed Money basis: `futures_only`.

Default markets: `WTI`, `BRENT`, `GASOIL`, `HEATOIL`, `RBOB`, `NATGAS`.

## 3. Contract selection and P&L

For each market and each trading date:

1. Identify the current nearby-1 contract's last-tradeable date from `contract_meta.csv`.
2. Define the roll date as the last business day of the month preceding that expiry.
3. Hold nearby 1 through the roll date and nearby 2 after the roll date.
4. Compute daily P&L only within the selected contract. Do not subtract prices across a contract switch.
5. Set cumulative P&L to the running sum of daily P&L.

The generic curve is used as a contract proxy. The selected nearby column is retained in the derived output for auditability.

## 4. Momentum features

Let $m$ index markets, $t$ index daily trading dates, and $n$ index lookback horizons. The market set has size $M=6$, and the horizon set is

$$
\mathcal{N}=\{5,10,15,\ldots,250\}, \qquad H=|\mathcal{N}|=50.
$$

For each market $m$, date $t$, and horizon $n\in\mathcal{N}$, define cumulative P&L and momentum:

$$
P_{m,t}=\sum_{i\leq t}dP_{m,i}
$$

$$
MOM_{m,t}(n)=\frac{P_{m,t}-MA_{m,t}(P;n)}{\sigma_{m,t}(P)}.
$$

The price/P&L volatility uses an exponentially weighted estimate with decay $\delta=60/61$ and annualization factor $\sqrt{252}$:

$$
\sigma_{m,t}(P)=\sqrt{252\,\frac{\displaystyle\sum_{i<t}\delta^i\left(dP_{m,t-i}\right)^2}{\displaystyle\sum_{i<t}\delta^i}}.
$$

For each market and date, the normalized feature vector is an $H$-dimensional column vector:

$$
\mathbf{x}_{m,t}=\begin{bmatrix}
 x_{m,t}(5)\\
 x_{m,t}(10)\\
 \vdots\\
 x_{m,t}(250)
\end{bmatrix}\in\mathbb{R}^{H},
 \qquad
 x_{m,t}(n)=\frac{MOM_{m,t}(n)}{\sigma_{m,t}(MOM)}.
$$

Here $\sigma_{m,t}(MOM)$ is the one-year rolling standard deviation of $MOM_{m,t}(n)$ for the same horizon. The resulting feature matrix for one daily date has shape $M\times H=6\times50$. After selecting the weekly Managed Money dates, the model input tensor has shape $T\times M\times H$, where $T$ is the number of usable weekly dates.

## 5. Dependent variable

For each weekly observation $t$, the dependent variable is the normalized Managed Money position:

$$
y_{m,t}=\frac{MM_{m,t}-MA_{m,t-5}(MM)}{\sigma_{m,t-5}(MM)}.
$$

For one weekly date, $\mathbf{y}_t=[y_{1,t},\ldots,y_{M,t}]^\mathsf{T}\in\mathbb{R}^{M}$. Across a training window of $T_w$ weekly dates, $Y$ has shape $T_w\times M$.

The moving average and standard deviation use the available weekly Managed Money observations for each market and are lagged by one reporting week. The lag prevents current Managed Money information from entering the predictor transformation.

## 6. Dimensioned neural-network model

### 6.1 Input and parameter dimensions

For one market-date pair, the input is $\mathbf{x}_{m,t}\in\mathbb{R}^{H}$, with $H=50$. The shared signal-generation layer contains $K=3$ latent factors. Its weight matrix is

$$
\mathbf{w}=\begin{bmatrix}
 w_{1,1}&w_{1,2}&\cdots&w_{1,H}\\
 w_{2,1}&w_{2,2}&\cdots&w_{2,H}\\
 w_{3,1}&w_{3,2}&\cdots&w_{3,H}
\end{bmatrix}\in\mathbb{R}^{K\times H}=\mathbb{R}^{3\times50}.
$$

There are $K\times H=150$ shared first-layer weights. The market-specific output weights are

$$
\mathbf{W}=\begin{bmatrix}
 W_{1}^{(1)}&W_{1}^{(2)}&W_{1}^{(3)}\\
 W_{2}^{(1)}&W_{2}^{(2)}&W_{2}^{(3)}\\
 \vdots&\vdots&\vdots\\
 W_{M}^{(1)}&W_{M}^{(2)}&W_{M}^{(3)}
\end{bmatrix}\in\mathbb{R}^{M\times K}=\mathbb{R}^{6\times3},
$$

which contributes $M\times K=18$ parameters. For a training window with $T_w$ weekly dates, the time-varying bias is

$$
\mathbf{B}=\left[b_{m,j}\right]_{m=1,j=1}^{M,T_w}\in\mathbb{R}^{M\times T_w}.
$$

Thus, before counting the bias values, the model has $150+18=168$ scalar weights. The bias contributes $M\times T_w$ additional scalar parameters for each rolling training window.

### 6.2 Shared signal-generation layer

For every market $m$, weekly date $t$, and latent factor $k$, first compute a scalar linear signal:

$$
a_{m,t}^{(k)}=\sum_{h=1}^{H}w_{k,h}x_{m,t,h}=\mathbf{w}_{k,:}\mathbf{x}_{m,t}\in\mathbb{R}.
$$

The same $\mathbf{w}\in\mathbb{R}^{K\times H}$ is used for all six markets. This is the cross-market shared signal-generation assumption.

Apply the reaction function elementwise:

$$
z_{m,t}^{(k)}=R\left(a_{m,t}^{(k)}\right),
\qquad
R(u)=u\exp\left(\frac{1-u^2}{2}\right).
$$

For one market-date pair, the latent score vector is

$$
\mathbf{z}_{m,t}=\begin{bmatrix}z_{m,t}^{(1)}\\z_{m,t}^{(2)}\\z_{m,t}^{(3)}\end{bmatrix}\in\mathbb{R}^{K}.
$$

For all market-date pairs, $\mathbf{Z}$ has shape $T_w\times M\times K$.

### 6.3 Market-specific output layer

The output weight vector for market $m$ is $\mathbf{W}_{m,:}\in\mathbb{R}^{K}$. The predicted normalized position is

$$
\widehat{y}_{m,t}=\mathbf{W}_{m,:}\mathbf{z}_{m,t}+b_{m,t}
=\sum_{k=1}^{K}W_m^{(k)}z_{m,t}^{(k)}+b_{m,t}.
$$

For one weekly date, $\widehat{\mathbf{y}}_t\in\mathbb{R}^{M}$. For a complete training window, $\widehat{Y}$ has shape $T_w\times M$. The trend-following component is

$$
CTA_{m,t}^{(z)}=\sum_{k=1}^{K}W_m^{(k)}z_{m,t}^{(k)},
$$

and the bias component is $b_{m,t}$. Their sum is the fitted normalized position.

### 6.4 Objective function and regularization

For a training window containing $T_w$ dates and $M$ markets, minimize

$$
\mathcal{L}(\theta)=
\frac{1}{MT_w}\sum_{m=1}^{M}\sum_{t=1}^{T_w}
\left(y_{m,t}-\widehat{y}_{m,t}\right)^2
+\lambda_1\sum_{k=1}^{K}\sum_{h=1}^{H}|w_{k,h}|
+\lambda_2\sum_{m=1}^{M}\sum_{j=2}^{T_w}
\left(b_{m,j}-b_{m,j-1}\right)^2.
$$

The parameter collection is

$$
\theta=\left(\mathbf{w},\mathbf{W},\mathbf{B}\right),
\qquad
\mathbf{w}\in\mathbb{R}^{3\times50},\quad
\mathbf{W}\in\mathbb{R}^{6\times3},\quad
\mathbf{B}\in\mathbb{R}^{6\times T_w}.
$$

The L1 term promotes sparsity in shared momentum horizons. The L2 term penalizes changes in each market's bias between adjacent weekly observations and prevents the bias from fitting every target observation independently.

Use PyTorch autograd and Adam. The model must be deterministic through fixed seeds, deterministic initialization, full-batch updates, and recorded package/runtime metadata.

## 7. Rolling evaluation

For each Tuesday evaluation date:

- Exclude the current observation from training.
- Use the preceding two years of weekly observations, denoted by $T_w$.
- Use 1,024 epochs for the first fit.
- Warm-start subsequent fits from the prior fitted parameters and use 36 epochs.
- Generate the current-date prediction without using the current Managed Money observation.

Primary reporting windows:

- Expanded history: all dates supported by the supplied inputs.
- Published comparison: evaluation dates in 2015-01-01 through 2025-12-31.

## 8. Forecasted position change

The model-implied weekly Managed Money change is

$$
\widehat{dMM}_{m,t}=
\sigma_{m,t-5}(MM)\left(\widehat{y}_{m,t}-\widehat{y}_{m,t-5}\right).
$$

The gradual movement of the slow component is excluded, matching the paper's evaluation definition.

## 9. Metrics

Pooled out-of-sample R-squared:

$$
R^2_{\mathrm{oos}}=
1-\frac{\displaystyle\sum_{m,t}\left(dMM_{m,t}-\widehat{dMM}_{m,t}\right)^2}
{\displaystyle\sum_{m,t}\left(dMM_{m,t}\right)^2}.
$$

Directional accuracy:

$$
Acc=\frac{1}{N}\sum_{m,t}
\mathbf{1}\!\left\{
\operatorname{sign}\left(dMM_{m,t}\right)=
\operatorname{sign}\left(\widehat{dMM}_{m,t}\right)
\right\}.
$$

Report pooled results, per-market results, and yearly results. Also report the benchmark results using the same observations and evaluation dates.

## 10. Validation checks

Before model fitting:

- Verify all six markets are represented.
- Verify no duplicate `(market, basis, report_week_tuesday)` rows after basis selection.
- Verify `mm_net == mm_long - mm_short` where the legs are available.
- Verify front/second nearby settlement availability over the evaluation sample.
- Verify roll dates are business days and selected nearby values are 1 or 2.
- Verify no training row is dated on or after its evaluation date.
- Verify feature columns have finite values after the warm-up period.

## 11. Outputs

Expected output groups:

- `derived/rolling_contract_daily.parquet`
- `derived/weekly_features.parquet`
- `outputs/benchmark_predictions.csv`
- `outputs/model_predictions.csv`
- `outputs/metrics_by_market_year.csv`
- `outputs/cta_discretionary_decomposition.csv`
- `outputs/run_manifest.json`
- diagnostic figures under `outputs/figures/`
